# Statistical traps that survive correct code

None of the code below has a bug. Every result is computed correctly and every result is
misleading. This is the second half of "attention to detail": after you have checked
the shifts and the merges, ask whether the *question* the numbers answer is the one you
think it is.

Each trap: the seductive result, the simulation or data slice that exposes it, and what
to say aloud.

**What's in here**
- Simpson's paradox · survivorship · regression to the mean
- selection on the outcome (best of N strategies) · multiple comparisons
- spurious correlation of trends · base-rate neglect
- aggregation traps (mean of ratios, unweighted percentages, volume-weighted prices)
- collider / Berkson bias · Goodhart and metric gaming
- "improvement" driven by one period · extrapolation outside the training range
- a 12-question checklist for any surprising result

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LinearRegression

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

hourly = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
meters["region"] = meters["region"].str.title()
readings = pd.read_csv("../data/meter_readings_daily.csv", parse_dates=["date"])
readings = readings[readings["meter_id"].isin(meters["meter_id"])]
print(hourly.shape, meters.shape, readings.shape)

(17520, 5) (300, 7) (107303, 3)


## 1. Simpson's paradox

A comparison that holds overall reverses inside every subgroup when the groups have
different mixes. Here: cost per kWh by region. SMEs pay a lower unit rate than
residential customers, and regions differ in their SME share.

The real meter data does not contain unit prices, so the tariff rates below are
simulated; the mix effect uses the actual regional customer-type shares.

In [2]:
annual = readings.groupby("meter_id")["kwh"].sum().rename("kwh")
m = meters.merge(annual, on="meter_id")
# simulated unit rates: SMEs cheaper per kWh; region A slightly cheaper than B within each type
rate = {"residential": 0.30, "sme": 0.18}
m["rate"] = m["customer_type"].map(rate)
m.loc[m["region"] == "London", "rate"] += 0.02          # London is MORE expensive within each type
m["cost"] = m["kwh"] * m["rate"]

sums = m.groupby("region")[["cost", "kwh"]].sum()
overall = (sums["cost"] / sums["kwh"]).rename("cost_per_kwh_overall")
within = m.pivot_table(index="region", columns="customer_type", values="cost", aggfunc="sum") / \
         m.pivot_table(index="region", columns="customer_type", values="kwh", aggfunc="sum")
mix = (m[m["customer_type"] == "sme"].groupby("region")["kwh"].sum() / sums["kwh"]).rename("sme_kwh_share")
pd.concat([overall, within.add_prefix("within_"), mix], axis=1).round(3).sort_values("cost_per_kwh_overall")

,cost_per_kwh_overall,within_residential,within_sme,sme_kwh_share
region,,,,
Midlands,0.231,0.30,0.18,0.574
Wales,0.232,0.30,0.18,0.568
North,0.237,0.30,0.18,0.523
London,0.260,0.32,0.20,0.503
Scotland,0.262,0.30,0.18,0.315


Within each customer type every region pays the same rate except London, which pays
more. Yet **Scotland** is the most expensive region overall: its SME share (the cheap
kWh) is only about 30% against 50-57% elsewhere. The overall ranking is a statement
about customer mix, not about tariffs. Always print the within-group table next to the
total before comparing groups.

**What to say aloud:** "Before I compare regions I want to know whether they have the
same customer mix. If not, I'll compare within customer type or standardise the mix."

## 2. Survivorship bias

"Average consumption per meter grew 4% in 2023" computed on meters that exist at the
end of 2023. Meters that churned mid-year are gone from the December sample, and
churners are not random: usage was falling before they left. Simulated below.

In [3]:
n = 2000
base = rng.lognormal(np.log(9), 0.4, n)                      # kWh/day in January
growth = rng.normal(0.0, 0.08, n)                            # true growth Jan -> Dec: mean 0 %
dec_all = base * (1 + growth)
churn_prob = 1 / (1 + np.exp(8 * growth + 1.5))              # falling usage -> more likely to churn
churned = rng.random(n) < churn_prob
survivors = ~churned
print(f"churn rate {churned.mean():.1%}")
print(f"true mean growth, all meters      : {growth.mean():+.2%}")
print(f"measured growth, survivors only    : {(dec_all[survivors].mean() / base[survivors].mean() - 1):+.2%}")
print(f"mean growth of churned (unobserved): {growth[churned].mean():+.2%}")

churn rate 20.2%
true mean growth, all meters      : +0.13%
measured growth, survivors only    : +1.19%
mean growth of churned (unobserved): -3.82%


**What to say aloud:** "Who is missing from this sample, and could the reason they are
missing be related to the thing I am measuring?"

## 3. Regression to the mean

Pick the 20 meters with the highest usage on one day. The next day they use less. No
intervention needed: the top 20 were selected partly because that day was unusually high
for them (noise), and noise does not repeat. Selecting on "unusually high relative to
their own average" makes the effect dramatic.

In [4]:
wide = readings.pivot(index="meter_id", columns="date", values="kwh")
d0, d1 = pd.Timestamp("2023-01-16"), pd.Timestamp("2023-01-17")
a, b = wide[d0].dropna(), wide[d1].dropna()
common = a.index.intersection(b.index); a, b = a[common], b[common]
top = a.nlargest(20).index                                         # highest absolute usage on day 0
own_mean = wide.loc[:, wide.columns.month == 1].mean(axis=1)[common]
top_rel = (a / own_mean).nlargest(20).index                        # most above their own January average
print(f"population      : {a.mean():6.2f} -> {b.mean():6.2f} kWh/day  ({b.mean()/a.mean()-1:+.1%})")
print(f"top-20 absolute : {a[top].mean():6.2f} -> {b[top].mean():6.2f} kWh/day  ({b[top].mean()/a[top].mean()-1:+.1%})")
print(f"top-20 relative : {a[top_rel].mean():6.2f} -> {b[top_rel].mean():6.2f} kWh/day  ({b[top_rel].mean()/a[top_rel].mean()-1:+.1%})")

population      :  20.89 ->  20.83 kWh/day  (-0.3%)
top-20 absolute : 116.61 -> 105.90 kWh/day  (-9.2%)
top-20 relative :  45.14 ->  28.09 kWh/day  (-37.8%)


The top group falls back toward the mean with no intervention at all, and the harder
you select on "unusual", the bigger the fake improvement. A campaign "targeting the heaviest users" that reports a 10%
reduction has to be compared with a control group selected the same way.

**What to say aloud:** "Was this group selected on the same variable we are now
measuring the change in?"

## 4. Selection on the outcome: best of N

Test 20 random trading rules on 2022, pick the best, report its 2022 Sharpe. The
expected maximum of 20 noise draws is about +1.9 standard errors. Its 2023 performance
is what you should have expected: nothing.

In [5]:
dp = hourly["price_eur_mwh"].diff().dropna()
y22, y23 = dp.loc["2022"], dp.loc["2023"]
n_rules = 20
rules = rng.choice([-1, 1], size=(n_rules, len(dp)))           # random signals, no information
def sharpe(sig, ret): 
    pnl = sig * ret.to_numpy(); return pnl.mean() / pnl.std() * np.sqrt(8760)
s22 = np.array([sharpe(rules[i, : len(y22)], y22) for i in range(n_rules)])
s23 = np.array([sharpe(rules[i, len(y22):], y23) for i in range(n_rules)])
best = s22.argmax()
print(f"best of {n_rules} rules on 2022: Sharpe {s22[best]:.2f}   same rule on 2023: {s23[best]:.2f}")
print(f"mean 2022 Sharpe across rules {s22.mean():+.2f}; expected max of {n_rules} N(0,1) draws ≈ {stats.norm.ppf(1 - 1/(n_rules+1)):.2f} SE")

best of 20 rules on 2022: Sharpe 2.15   same rule on 2023: -0.67
mean 2022 Sharpe across rules +0.33; expected max of 20 N(0,1) draws ≈ 1.67 SE


**What to say aloud:** "How many things were tried before this one was chosen, and was
the number I am shown computed on the data used to choose it?"

## 5. Multiple comparisons in feature search

Add 50 pure-noise features to a regression and rank them by |t|. Some are "significant"
at 5% by construction (about 2.5 of 50). The mechanics of Bonferroni / Benjamini-Hochberg
are in `03_scipy/04_hypothesis_tests_power_and_pitfalls.ipynb`; here just the
size of the effect.

In [6]:
y = hourly["consumption_mwh"].loc["2023"].to_numpy()
y = y - y.mean()
Z = rng.normal(size=(len(y), 50))
X = np.column_stack([np.ones(len(y)), Z])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
e = y - X @ beta
se = np.sqrt((e @ e) / (len(y) - X.shape[1]) * np.diag(np.linalg.inv(X.T @ X)))
t = beta / se
p = 2 * (1 - stats.t.cdf(np.abs(t[1:]), df=len(y) - X.shape[1]))
print("noise features with p < 0.05:", (p < 0.05).sum(), "of 50 | smallest p:", p.min().round(4))
print("R² of noise-only model:", round(1 - (e @ e) / (y @ y), 4))

noise features with p < 0.05: 3 of 50 | smallest p: 0.0219
R² of noise-only model: 0.0057


## 6. Spurious correlation of trends (brief)

Covered in `04_classical_time_series_stats.ipynb` §8: two unrelated trending series
correlate strongly in levels and not at all in differences. Any 2022 correlation with
price must be re-checked in changes or after detrending.

## 7. Base-rate neglect

A spike detector with 95% sensitivity and 95% specificity applied to hours where the
base rate of spikes is 2%: most alarms are false. Precision comes from Bayes, not from
accuracy.

In [7]:
sens, spec, base_rate = 0.95, 0.95, 0.02
tp = sens * base_rate; fp = (1 - spec) * (1 - base_rate)
precision = tp / (tp + fp)
print(f"precision (share of alarms that are real spikes): {precision:.1%}")
print(f"accuracy of the detector: {sens*base_rate + spec*(1-base_rate):.1%}   accuracy of 'never alarm': {1-base_rate:.1%}")
for br in [0.005, 0.02, 0.10, 0.30]:
    tp = sens * br; fp = (1 - spec) * (1 - br)
    print(f"  base rate {br:5.1%} -> precision {tp/(tp+fp):.1%}")

precision (share of alarms that are real spikes): 27.9%
accuracy of the detector: 95.0%   accuracy of 'never alarm': 98.0%
  base rate  0.5% -> precision 8.7%
  base rate  2.0% -> precision 27.9%
  base rate 10.0% -> precision 67.9%
  base rate 30.0% -> precision 89.1%


**What to say aloud:** "What is the base rate, and what is precision at the threshold we
would actually use?"

## 8. Aggregation traps

Three ways to get a different "average price" from the same data. Only one of them is
what a retailer paid.

In [8]:
h = hourly.loc["2023"]
share = 0.01
cost = h["price_eur_mwh"] * h["consumption_mwh"] * share
per_hour_ratio = (cost / (h["consumption_mwh"] * share))                     # = price
print(f"time-weighted mean price      : {h['price_eur_mwh'].mean():.2f}")
print(f"volume-weighted (paid) price  : {cost.sum() / (h['consumption_mwh'] * share).sum():.2f}")
daily_ratio = (cost.resample('D').sum() / (h['consumption_mwh'] * share).resample('D').sum())
print(f"mean of daily cost/volume     : {daily_ratio.mean():.2f}   (mean of ratios ≠ ratio of sums)")

time-weighted mean price      : 84.76
volume-weighted (paid) price  : 88.38
mean of daily cost/volume     : 86.98   (mean of ratios ≠ ratio of sums)


In [9]:
# averaging percentages across groups of different size
g = pd.DataFrame({"region": ["A", "B", "C"], "meters": [900, 50, 50], "solar_share": [0.05, 0.40, 0.45]})
print("unweighted mean of regional solar shares:", round(g["solar_share"].mean(), 3))
print("actual share of meters with solar        :", round((g["meters"] * g["solar_share"]).sum() / g["meters"].sum(), 3))

unweighted mean of regional solar shares: 0.3
actual share of meters with solar        : 0.088


**What to say aloud:** "Is this a mean of ratios or a ratio of sums, and which one does
the business question need?"

## 9. Collider / Berkson bias

Condition on something caused by both variables and you create a correlation that does
not exist. Simulation: complaints happen when bills are high **or** service is poor.
Among complainers, bill size and poor service look *negatively* related (equivalently,
big bills go with *better* service) although they are independent in the population. An
analyst who only sees the complaints file concludes "large customers get better treatment".

In [10]:
n = 20000
bill = rng.normal(size=n); poor_service = rng.normal(size=n)          # independent
complained = (bill > 1) | (poor_service > 1)
print(f"corr(bill, poor_service) in population   : {np.corrcoef(bill, poor_service)[0,1]:+.3f}")
print(f"corr(bill, poor_service) among complainers: {np.corrcoef(bill[complained], poor_service[complained])[0,1]:+.3f}  (n={complained.sum()})")

corr(bill, poor_service) in population   : +0.010
corr(bill, poor_service) among complainers: -0.561  (n=5752)


**What to say aloud:** "Is the sample I'm analysing defined by something that both
variables influence?"

## 10. Goodhart: optimise a metric, get the metric

Minimising MAPE rewards forecasting **low** (a 50% under-forecast costs at most 50%, a
50% over-forecast costs 50% of a smaller denominator, so errors are asymmetric in the
ratio). Maximising directional accuracy ignores magnitude entirely.

In [11]:
y = hourly["price_eur_mwh"].loc["2023"]
y = y[y > 0].to_numpy()                      # MAPE needs a positive denominator
true_mean = y.mean()
def mape(c): return np.mean(np.abs(y - c) / y)
def rmse(c): return np.sqrt(np.mean((y - c) ** 2))
c_mape = minimize_scalar(mape, bounds=(y.min(), y.max()), method="bounded").x
c_rmse = minimize_scalar(rmse, bounds=(y.min(), y.max()), method="bounded").x
print(f"constant that minimises MAPE: {c_mape:,.1f}   RMSE-optimal (the mean): {c_rmse:,.1f}   median {np.median(y):.1f}   bias of the MAPE choice: {c_mape/true_mean-1:+.1%}")

constant that minimises MAPE: 62.4   RMSE-optimal (the mean): 85.1   median 83.6   bias of the MAPE choice: -26.7%


In [12]:
# directional accuracy: a forecaster that nails the sign of quiet hours and misses every spike (simulated)
n = 8760
spike = rng.random(n) < 0.10
moves = np.where(spike, rng.normal(0, 25, n), rng.normal(0, 2, n))     # 10% of hours carry the big moves
signal = np.sign(moves) * np.where(spike, -1, 1)                       # right on quiet hours, wrong on spikes
pnl = signal * moves
print(f"directional accuracy {np.mean(np.sign(signal) == np.sign(moves)):.0%}, but P&L per hour {pnl.mean():+.2f} EUR/MWh")

directional accuracy 90%, but P&L per hour -0.58 EUR/MWh


**What to say aloud:** "Which decision does this metric stand in for, and can the metric
be improved without improving the decision?"

## 11. The improvement that is one period

Model B beats A over the year. Per month, A wins most of the time; B's edge is one
regime. Same trick in reverse hides a genuinely better model behind one bad month.

In [13]:
months = pd.period_range("2023-01", "2023-12", freq="M")
err_a = rng.normal(100, 5, 12)
err_b = err_a + rng.normal(3, 1, 12)                    # B is worse in every typical month
err_b[6] = err_a[6] - 45                                 # ... except one month where it is far better
tbl = pd.DataFrame({"rmse_A": err_a, "rmse_B": err_b}, index=months).round(1)
print(f"annual RMSE: A {np.sqrt((err_a**2).mean()):.1f}   B {np.sqrt((err_b**2).mean()):.1f}   -> B 'wins'")
print(f"months where A is better: {(err_a < err_b).sum()} of 12")
tbl.T

annual RMSE: A 98.7   B 98.8   -> B 'wins'
months where A is better: 11 of 12


,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
rmse_A,104.9,98.9,96.4,99.1,97.6,96.9,94.3,95.4,97.5,103.3,99.0,100.9
rmse_B,107.8,102.1,98.3,102.8,101.2,99.3,49.3,98.6,100.7,108.4,100.2,103.5


**What to say aloud:** "Show me the per-period wins, not just the total. Is the gain
concentrated, and do I expect that regime to recur?"

## 12. Extrapolation outside the training range

Fit demand on temperature using 2022 (max temperature seen ≈ 28 °C) and apply it to a
heatwave at 35 °C. A quadratic extrapolates to nonsense; a piecewise-linear model at
least extrapolates its last slope. Neither has seen 35 °C.

In [14]:
h22 = hourly.loc["2022"]
t = h22["temp_c"].to_numpy(); y = h22["consumption_mwh"].to_numpy()
print(f"training temperature range: {t.min():.1f} to {t.max():.1f} °C")
Xq = np.column_stack([np.ones_like(t), t, t ** 2, t ** 3])
bq, *_ = np.linalg.lstsq(Xq, y, rcond=None)
hdd, cdd = np.clip(15 - t, 0, None), np.clip(t - 22, 0, None)
Xp = np.column_stack([np.ones_like(t), hdd, cdd]); bp, *_ = np.linalg.lstsq(Xp, y, rcond=None)
for T in [25, 30, 35, 40]:
    cubic = bq @ [1, T, T ** 2, T ** 3]
    piece = bp @ [1, max(15 - T, 0), max(T - 22, 0)]
    print(f"T={T} °C  cubic {cubic:8,.0f}   piecewise {piece:8,.0f}   (mean load {y.mean():,.0f})")

training temperature range: -6.2 to 27.7 °C
T=25 °C  cubic   33,343   piecewise   29,645   (mean load 29,407)
T=30 °C  cubic   43,990   piecewise   32,217   (mean load 29,407)
T=35 °C  cubic   62,283   piecewise   34,790   (mean load 29,407)
T=40 °C  cubic   90,043   piecewise   37,363   (mean load 29,407)


**What to say aloud:** "Where is this input relative to the training range? If it is
outside, the model's answer is an assumption, not an estimate."

## Questions to ask about any surprising result

1. What is one observation, and how many are there really (after duplicates, after autocorrelation)?
2. Who or what is missing from the sample, and is the reason related to the outcome?
3. Was this group selected on the variable I am now measuring?
4. How many alternatives were tried before this one was reported?
5. Was the number computed on the data used to choose the model?
6. Do the subgroups agree with the total? If not, is the mix different?
7. Is this a mean of ratios or a ratio of sums? Weighted by what?
8. Does the comparison hold period by period, or is it one regime?
9. What is the base rate, and what is precision at the operating threshold?
10. Is the input inside the training range?
11. Is the metric the decision, or a proxy that can be gamed?
12. What would I expect to see if there were no effect at all, and is this distinguishable from it?